## Importing Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    LinearSVC
)

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

from pyspark.ml.tuning import (
    CrossValidator,
    ParamGridBuilder
)

import pandas as pd
import time
import builtins

## Creating Spark Session

In [2]:
spark = SparkSession.builder \
    .appName("Task3_Classification") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

## Loading Dataset

In [3]:
df = spark.read.parquet("nifty_10m_fixed.parquet")

## Dataset Overview

In [4]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

df.printSchema()

df.show(5)

Rows : 10000000
Columns : 15
root
 |-- timestamp: string (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- iv: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- oi: long (nullable = true)
 |-- strike_price: double (nullable = true)
 |-- spot: double (nullable = true)
 |-- datetime: string (nullable = true)
 |-- date: string (nullable = true)
 |-- expiry_type: string (nullable = true)
 |-- strike_type: string (nullable = true)
 |-- option_type: string (nullable = true)

+----------+----+----+----+-----+---------+------+-----+------------+----------+-------------------+----------+-----------+-----------+-----------+
| timestamp|open|high| low|close|       iv|volume|   oi|strike_price|      spot|           datetime|      date|expiry_type|strike_type|option_type|
+----------+----+----+----+-----+---------+------+-----+------------+----------+-----------------

## Creating Classification Target

In [5]:
df = df.withColumn(
    "label",
    when(col("close") > col("open"), 1).otherwise(0)
)

## Checking Class Balance

In [6]:
df.groupBy("label").count().show()

+-----+-------+
|label|  count|
+-----+-------+
|    1|4669165|
|    0|5330835|
+-----+-------+



## Selecting Features

In [7]:
numerical = [

    "open",

    "high",

    "low",

    "iv",

    "volume",

    "oi",

    "strike_price",

    "spot"

]

categorical = [

    "expiry_type",

    "strike_type",

    "option_type"

]

## Encoding Categories

In [8]:
indexers = [

    StringIndexer(

        inputCol=c,

        outputCol=c+"_index",

        handleInvalid="keep"

    )

    for c in categorical

]

## One Hot Encoding

In [9]:
encoder = OneHotEncoder(

    inputCols=[c+"_index" for c in categorical],

    outputCols=[c+"_vec" for c in categorical]

)

## Assemble Numerical Features

In [10]:
assembler = VectorAssembler(

    inputCols=numerical,

    outputCol="num_features"

)

## Scaling

In [11]:
scaler = StandardScaler(

    inputCol="num_features",

    outputCol="scaled_num"

)

## Final Feature Vector

In [12]:
assembler2 = VectorAssembler(

    inputCols=[

        "scaled_num",

        "expiry_type_vec",

        "strike_type_vec",

        "option_type_vec"

    ],

    outputCol="features"

)

## Pipeline

In [13]:
pipeline = Pipeline(

    stages=

        indexers +

        [

            encoder,

            assembler,

            scaler,

            assembler2

        ]

)

In [14]:
model = pipeline.fit(df)

data = model.transform(df)

In [15]:
data = data.select(

    "features",

    "label"

)

## Train-Test Splitting

In [16]:
train, test = data.randomSplit(

    [0.8,0.2],

    seed=42

)

## Cache Dataset

In [17]:
train.cache()

test.cache()

train.count()

test.count()

1999828

## Classification Metrics


In [18]:
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

results = []

In [19]:
def evaluate_model(model_name, model, train, test):

    print("="*60)
    print(f"Training {model_name}...")
    print("="*60)

    start = time.time()

    fitted_model = model.fit(train)

    end = time.time()

    training_time = end - start

    prediction = fitted_model.transform(test)

    accuracy = accuracy_eval.evaluate(prediction)
    precision = precision_eval.evaluate(prediction)
    recall = recall_eval.evaluate(prediction)
    f1 = f1_eval.evaluate(prediction)

    try:
        auc = auc_eval.evaluate(prediction)
    except:
        auc = None

    # Print Results

    print(f"\nModel : {model_name}")
    print(f"Training Time (s) : {builtins.round(training_time,2)}")
    print(f"Accuracy          : {builtins.round(accuracy,4)}")
    print(f"Precision         : {builtins.round(precision,4)}")
    print(f"Recall            : {builtins.round(recall,4)}")
    print(f"F1 Score          : {builtins.round(f1,4)}")

    if auc is not None:
        print(f"AUC-ROC           : {builtins.round(auc,4)}")
    else:
        print("AUC-ROC           : N/A")

    print("="*60)

    results.append({

        "Model":model_name,

        "Training Time (s)":builtins.round(training_time,2),

        "Accuracy":builtins.round(accuracy,4),

        "Precision":builtins.round(precision,4),

        "Recall":builtins.round(recall,4),

        "F1 Score":builtins.round(f1,4),

        "AUC-ROC":"N/A" if auc is None else builtins.round(auc,4)

    })

    return fitted_model

## Logistic Regression

In [20]:
lr = LogisticRegression(

    featuresCol="features",

    labelCol="label",

    maxIter=20,

    regParam=0.01

)

lr_model = evaluate_model(

    "Logistic Regression",

    lr,

    train,

    test

)

Training Logistic Regression...

Model : Logistic Regression
Training Time (s) : 60.72
Accuracy          : 0.5387
Precision         : 0.5367
Recall            : 0.5387
F1 Score          : 0.4358
AUC-ROC           : 0.5407


## Decision Tree

In [21]:
dt = DecisionTreeClassifier(

    featuresCol="features",

    labelCol="label",

    maxDepth=5,

    maxBins=32

)

dt_model = evaluate_model(

    "Decision Tree",

    dt,

    train,

    test

)

Training Decision Tree...

Model : Decision Tree
Training Time (s) : 54.42
Accuracy          : 0.5339
Precision         : 0.664
Recall            : 0.5339
F1 Score          : 0.3726
AUC-ROC           : 0.5389


## Random Forest

In [22]:
rf = RandomForestClassifier(

    featuresCol="features",

    labelCol="label",

    numTrees=20,

    maxDepth=8,

    seed=42

)

rf_model = evaluate_model(

    "Random Forest",

    rf,

    train,

    test

)

Training Random Forest...

Model : Random Forest
Training Time (s) : 296.04
Accuracy          : 0.538
Precision         : 0.7011
Recall            : 0.538
F1 Score          : 0.3825
AUC-ROC           : 0.5586


## Linear SVM

In [23]:
svm = LinearSVC(

    featuresCol="features",

    labelCol="label",

    maxIter=20,

    regParam=0.01

)

svm_model = evaluate_model(

    "Linear SVM",

    svm,

    train,

    test

)

Training Linear SVM...

Model : Linear SVM
Training Time (s) : 69.91
Accuracy          : 0.5334
Precision         : 0.5266
Recall            : 0.5334
F1 Score          : 0.3712
AUC-ROC           : 0.5392


## Baseline Results

In [24]:
baseline_results = pd.DataFrame(results)

baseline_results

,Model,Training Time (s),Accuracy,Precision,Recall,F1 Score,AUC-ROC
0,Logistic Regression,60.72,0.5387,0.5367,0.5387,0.4358,0.5407
1,Decision Tree,54.42,0.5339,0.6640,0.5339,0.3726,0.5389
2,Random Forest,296.04,0.5380,0.7011,0.5380,0.3825,0.5586
3,Linear SVM,69.91,0.5334,0.5266,0.5334,0.3712,0.5392


In [25]:
tuned_results = []

## Hyperparameter Tuning

In [26]:
def tune_model(model_name, estimator, paramGrid, train, test, evaluator):

    print("="*70)
    print(f"Hyperparameter Tuning : {model_name}")
    print("="*70)

    cv = CrossValidator(
        estimator=estimator,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        numFolds=2,
        parallelism=2
    )

    start = time.time()

    cvModel = cv.fit(train)

    end = time.time()

    bestModel = cvModel.bestModel

    prediction = bestModel.transform(test)

    accuracy = accuracy_eval.evaluate(prediction)
    precision = precision_eval.evaluate(prediction)
    recall = recall_eval.evaluate(prediction)
    f1 = f1_eval.evaluate(prediction)

    try:
        auc = auc_eval.evaluate(prediction)
    except:
        auc = None

    print("\nBest Model Parameters")

    print(bestModel.extractParamMap())

    print("\nTraining Time :", builtins.round(end-start,2),"seconds")
    print("Accuracy      :", builtins.round(accuracy,4))
    print("Precision     :", builtins.round(precision,4))
    print("Recall        :", builtins.round(recall,4))
    print("F1 Score      :", builtins.round(f1,4))

    if auc is not None:
        print("AUC ROC       :", builtins.round(auc,4))
    else:
        print("AUC ROC       : N/A")

    tuned_results.append({

        "Model":model_name,

        "Training Time (s)":builtins.round(end-start,2),

        "Accuracy":builtins.round(accuracy,4),

        "Precision":builtins.round(precision,4),

        "Recall":builtins.round(recall,4),

        "F1 Score":builtins.round(f1,4),

        "AUC-ROC":"N/A" if auc is None else builtins.round(auc,4)

    })

    return cvModel

In [28]:
import os

# Creating Folder
os.makedirs("Saved_Models", exist_ok=True)

# Saving Models

lr_model.write().overwrite().save("Saved_Models/Logistic_Regression")
dt_model.write().overwrite().save("Saved_Models/Decision_Tree")
rf_model.write().overwrite().save("Saved_Models/Random_Forest")
svm_model.write().overwrite().save("Saved_Models/Linear_SVM")

# Saving Predictions

pred_lr = lr_model.transform(test)
pred_dt = dt_model.transform(test)
pred_rf = rf_model.transform(test)
pred_svm = svm_model.transform(test)

pred_lr.select(
    "label",
    "prediction",
    "probability",
    "rawPrediction"
).write.mode("overwrite").parquet(
    "Saved_Models/LR_Predictions"
)

pred_dt.select(
    "label",
    "prediction",
    "probability",
    "rawPrediction"
).write.mode("overwrite").parquet(
    "Saved_Models/DT_Predictions"
)

pred_rf.select(
    "label",
    "prediction",
    "probability",
    "rawPrediction"
).write.mode("overwrite").parquet(
    "Saved_Models/RF_Predictions"
)

pred_svm.select(
    "label",
    "prediction",
    "rawPrediction"
).write.mode("overwrite").parquet(
    "Saved_Models/SVM_Predictions"
)

# Saving Metrics

baseline_results.to_csv(
    "Saved_Models/Model_Metrics.csv",
    index=False
)

In [29]:
import shutil

# Create ZIP file
shutil.make_archive(
    "Saved_Models",
    "zip",
    "Saved_Models"
)

print("ZIP file created successfully.")

ZIP file created successfully.


In [30]:
from google.colab import files

files.download("Saved_Models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>